# LightGBM 实验 — **方式1：按标的分块 + 多模型**

本 notebook：

1. 用 `scan_parquet` 识别列与行数，**不常驻全量宽表**。
2. 将 `valid_tickers_full` 按 `CHUNK_SIZE` 分批：`read_parquet(columns=...)` → `long_df` → `feat_df` → 训练 **一个 LGBM / 批**，保存到 `artifacts/way1_ensemble/`。
3. 拼接各批在验证区间的预测得到 `ensemble_backtest_df`，供回测格使用。

与 **方式2（单模型 `init_model` 续训）** 对照见：`lgb_exp_way2_init_model.ipynb`。



In [1]:
from pathlib import Path
import polars as pl

data_path = Path("train.parquet")
if not data_path.exists():
    raise FileNotFoundError("未找到 train.parquet，请先将文件放在当前项目目录下。")

lf = pl.scan_parquet(data_path)
col_names = lf.collect_schema().names()
n_rows = lf.select(pl.len()).collect().item()
print(f"train.parquet 行数={n_rows:,} 列数={len(col_names)}")
print("方式1：不加载全量宽表；训练按批读取列子集。")
col_names[:20]



In [2]:
# 解析宽表列名 -> (ticker, close/volume)，得到全市场标的列表
from lgb_data_pipeline import discover_pairs

if "col_names" not in globals():
    raise ValueError("请先运行上一格。")

pairs = discover_pairs(col_names)
valid_tickers_full = sorted([t for t, d in pairs.items() if "close" in d and "volume" in d])
print(f"总标的数量（同时有 close/volume 列）: {len(valid_tickers_full)}")



### 说明

- **方式1**：每批标的独立训练一个 `LGBMRegressor`（各自早停），磁盘保存 `chunk_XXX.pkl`；验证集预测行合并为 `ensemble_backtest_df`（标的互不重叠，无需再平均）。
- 下一格打印分块数量；再下一格为「单批特征预览」（可选）；主训练在随后一格。



In [ ]:
# 分块规划
CHUNK_SIZE = 30  # 与训练格中保持一致；OOM 时改小

if "valid_tickers_full" not in globals():
    raise ValueError("请先运行上一格。")

n_full = len(valid_tickers_full)
n_batches = (n_full + CHUNK_SIZE - 1) // CHUNK_SIZE
print(f"全量标的 {n_full} 只：每批 {CHUNK_SIZE} 只 → 共 {n_batches} 批（{n_batches} 个子模型）。")



In [3]:
#### （可选）单批特征预览

若需像旧版一样查看 `long_df` / `feat_df` 头部：将 `PREVIEW_BATCH=0` 保留，运行下一格；否则将下一格跳过。


In [ ]:
# 可选：仅第一批，查看特征形状（主训练不依赖本格）
import gc
import polars as pl
from lgb_data_pipeline import wide_columns_for_tickers, wide_to_long_with_mo, build_feat_df

PREVIEW_BATCH = True  # 设为 False 可跳过本格
if not PREVIEW_BATCH:
    print("已跳过预览。")
else:
    batch = valid_tickers_full[:CHUNK_SIZE]
    cols = wide_columns_for_tickers(pairs, batch)
    df_w = pl.read_parquet(data_path, columns=cols)
    long_df = wide_to_long_with_mo(df_w, pairs, batch)
    del df_w
    gc.collect()
    feat_df = build_feat_df(long_df)
    print("long_df", long_df.shape, "feat_df", feat_df.shape)
    print(feat_df.head(3))
    del long_df, feat_df
    gc.collect()



In [ ]:
# 方式1：按批训练多个模型 + 合并验证集预测
from pathlib import Path
import gc
import joblib
import numpy as np
import polars as pl
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, r2_score

from lgb_data_pipeline import (
    wide_columns_for_tickers,
    wide_to_long_with_mo,
    build_feat_df,
    global_split_time_from_parquet,
)

HORIZON = 1
MAX_BATCHES = None  # 设为 1 做冒烟测试；None 训满所有批

split_time = global_split_time_from_parquet(data_path, 0.8)
print(f"全局切分时刻（全表 datetime 排序 80% 行）: {split_time}")

batches = [
    valid_tickers_full[i : i + CHUNK_SIZE]
    for i in range(0, len(valid_tickers_full), CHUNK_SIZE)
]
if MAX_BATCHES is not None:
    batches = batches[: int(MAX_BATCHES)]
print(f"训练批次数: {len(batches)}")

out_dir = Path("artifacts/way1_ensemble")
out_dir.mkdir(parents=True, exist_ok=True)

valid_pred_parts = []
feature_cols_ref = None
last_lgb_model = None

base_exclude = {
    "datetime", "ticker", "close", "volume", "target_fwd_5m",
    "return_5m_lag1", "close_ma", "volume_ma", "volume_diff", "volume_diff_std",
    "close_change_rate", "close_ma_24h", "volume_ma_24h", "close_volume_corr",
    "close_volume_corr_mean", "close_volume_corr_std", "upward_mean_past_24h",
    "downward_mean_past_24h", "rs_past_24h",
}
high_freq_keep = {
    "return_5m", "volume_change_rate", "volume_change_rate_change_rate",
    "price_change_change_rate", "price_volume_direction",
    "price_breakout_confirmation", "volume_breakout_confirmation",
}
numeric_dtypes = (pl.Float32, pl.Float64, pl.Int8, pl.Int16, pl.Int32, pl.Int64)

for bi, batch in enumerate(batches):
    print(f"\n>>> 批次 {bi + 1}/{len(batches)} 标的数={len(batch)}")
    cols = wide_columns_for_tickers(pairs, batch)
    df_w = pl.read_parquet(data_path, columns=cols)
    long_df = wide_to_long_with_mo(df_w, pairs, batch)
    del df_w
    gc.collect()

    feat_df = build_feat_df(long_df)
    del long_df
    gc.collect()

    model_df = feat_df.sort(["ticker", "datetime"]).with_columns([
        (pl.col("close").shift(-HORIZON).over("ticker") / (pl.col("close") + 1e-12) - 1).alias("target_fwd_5m"),
    ])
    del feat_df
    gc.collect()

    all_numeric_cols = [c for c, dt in model_df.schema.items() if dt in numeric_dtypes]
    ewm_cols = [c for c in all_numeric_cols if c.endswith("_ewm3d")]
    hf_cols = [c for c in all_numeric_cols if c in high_freq_keep]
    feature_cols = sorted(set(ewm_cols + hf_cols) - {"target_fwd_5m"})
    if len(feature_cols) == 0:
        feature_cols = [c for c in all_numeric_cols if c not in base_exclude]

    if feature_cols_ref is None:
        feature_cols_ref = feature_cols
    elif feature_cols != feature_cols_ref:
        raise ValueError(f"批次 {bi} 特征列与首批不一致")

    clean_df = model_df.with_columns([
        pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
        for c in feature_cols + ["target_fwd_5m"]
    ])
    clean_df = clean_df.filter(pl.col("target_fwd_5m").is_not_null())
    clean_df = clean_df.with_columns([
        pl.col(c).fill_null(pl.col(c).median().over("ticker")).alias(c) for c in feature_cols
    ])
    clean_df = clean_df.with_columns([
        pl.col(c).fill_null(pl.col(c).median()).fill_null(0.0).cast(pl.Float32).alias(c)
        for c in feature_cols
    ])

    train_df = clean_df.filter(pl.col("datetime") <= split_time)
    valid_df = clean_df.filter(pl.col("datetime") > split_time)
    del clean_df, model_df
    gc.collect()

    print(f"    训练行 {train_df.height:,} | 验证行 {valid_df.height:,}")

    X_train = np.ascontiguousarray(train_df.select(feature_cols).to_numpy(), dtype=np.float32)
    y_train = train_df.select("target_fwd_5m").to_numpy().ravel().astype(np.float32, copy=False)
    X_valid = np.ascontiguousarray(valid_df.select(feature_cols).to_numpy(), dtype=np.float32)
    y_valid = valid_df.select("target_fwd_5m").to_numpy().ravel().astype(np.float32, copy=False)
    del train_df
    gc.collect()

    lgb_model = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=1200,
        learning_rate=0.03,
        num_leaves=63,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.5,
        random_state=42,
        n_jobs=-1,
    )
    lgb_model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_names=["valid"],
        eval_metric="l2",
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)],
    )
    del X_train, y_train
    gc.collect()

    y_pred = lgb_model.predict(X_valid, num_iteration=lgb_model.best_iteration_)
    rmse = mean_squared_error(y_valid, y_pred) ** 0.5
    r2 = r2_score(y_valid, y_pred)
    print(f"    Valid RMSE={rmse:.8f} R2={r2:.6f} best_iter={lgb_model.best_iteration_}")

    valid_pred_parts.append(
        valid_df.select(["datetime", "ticker", "target_fwd_5m"]).with_columns(
            pl.Series("pred_ret_5m", y_pred)
        )
    )

    joblib.dump(
        {
            "model": lgb_model,
            "feature_cols": feature_cols,
            "tickers": batch,
            "split_time": str(split_time),
            "best_iteration": int(lgb_model.best_iteration_) if getattr(lgb_model, "best_iteration_", None) else None,
            "chunk_id": bi,
        },
        out_dir / f"chunk_{bi:03d}.pkl",
    )

    last_lgb_model = lgb_model
    del X_valid, y_valid, valid_df, lgb_model
    gc.collect()

feature_cols = feature_cols_ref
lgb_model = last_lgb_model
WAY1_ENSEMBLE = True
ensemble_backtest_df = pl.concat(valid_pred_parts).sort(["datetime", "ticker"])
print(f"\n合并后验证预测行数: {ensemble_backtest_df.height:,} ；子模型目录: {out_dir.resolve()}")



In [ ]:
# 回测：方式1 用 ensemble_backtest_df；否则用 valid_df + 单模型
import polars as pl
import matplotlib.pyplot as plt

if globals().get("WAY1_ENSEMBLE"):
    if "ensemble_backtest_df" not in globals():
        raise ValueError("请先运行方式1 训练格。")
    pred_df = ensemble_backtest_df
else:
    if "lgb_model" not in globals() or "feature_cols" not in globals() or "valid_df" not in globals():
        raise ValueError("请先运行训练 cell。")
    pred_df = valid_df.with_columns([
        pl.Series(
            "pred_ret_5m",
            lgb_model.predict(
                valid_df.select(feature_cols).to_numpy(),
                num_iteration=lgb_model.best_iteration_,
            ),
        )
    ])

TOP_K = 1

pred_df = pred_df.with_columns([
    pl.col("pred_ret_5m").rank("dense", descending=True).over("datetime").alias("pred_rank"),
])

port_df = (
    pred_df.filter(pl.col("pred_rank") <= TOP_K)
    .group_by("datetime")
    .agg([
        pl.col("target_fwd_5m").mean().alias("portfolio_ret_5m"),
        pl.len().alias("n_holdings"),
    ])
    .sort("datetime")
)

port_df = port_df.with_columns([
    (pl.col("portfolio_ret_5m") + 1.0).cum_prod().alias("equity_curve"),
])

stats = port_df.select([
    pl.col("portfolio_ret_5m").mean().alias("mean_ret_5m"),
    pl.col("portfolio_ret_5m").std().alias("std_ret_5m"),
    pl.col("portfolio_ret_5m").sum().alias("sum_ret_5m"),
    pl.col("equity_curve").last().alias("final_nav"),
    pl.col("n_holdings").mean().alias("avg_holdings"),
]).to_dicts()[0]

print("回测统计（验证区间）")
for k, v in stats.items():
    print(f"{k}: {v}")

plot_pd = port_df.select(["datetime", "equity_curve"]).to_pandas()
plt.figure(figsize=(12, 4))
plt.plot(plot_pd["datetime"], plot_pd["equity_curve"], label=f"Top{TOP_K} Equal-Weight Long")
plt.title("Equity Curve (5m)")
plt.xlabel("Datetime")
plt.ylabel("Net Value")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()



In [ ]:
# 保存：方式1 写入 ensemble 目录说明；单模型仍写原路径
from pathlib import Path
import joblib

model_dir = Path("artifacts")
model_dir.mkdir(parents=True, exist_ok=True)

if globals().get("WAY1_ENSEMBLE"):
    meta = {
        "mode": "way1_ensemble",
        "ensemble_dir": str((model_dir / "way1_ensemble").resolve()),
        "feature_cols": feature_cols,
        "n_chunks": len(list((model_dir / "way1_ensemble").glob("chunk_*.pkl"))),
    }
    joblib.dump(meta, model_dir / "lgb_way1_meta.pkl")
    print(f"已保存方式1 元数据: {model_dir / 'lgb_way1_meta.pkl'}")
else:
    if "lgb_model" not in globals() or "feature_cols" not in globals():
        raise ValueError("请先运行训练 cell。")
    model_bundle = {
        "model": lgb_model,
        "feature_cols": feature_cols,
        "best_iteration": int(lgb_model.best_iteration_) if getattr(lgb_model, "best_iteration_", None) else None,
    }
    joblib.dump(model_bundle, model_dir / "lgb_model_5m.pkl")
    print(f"模型已保存: {(model_dir / 'lgb_model_5m.pkl').resolve()}")



In [ ]:
# [A] validation 特征工程（单独 cell）
from pathlib import Path
import re
import polars as pl

val_path = Path("validation.parquet")
if not val_path.exists():
    raise FileNotFoundError("未找到 validation.parquet，请先放到当前目录。")

val_raw = pl.read_parquet(val_path)
print(f"validation shape: {val_raw.shape}")

# 宽表 -> long（内存友好）
MAX_TICKERS_VAL = None  # 内存紧张可改成 50
pat_tuple = re.compile(r"^\('([^']+)'\s*,\s*'([^']+)'\)$")
pat_flat = re.compile(r"^(.*?)_([^_]+)$")

pairs = {}
for c in val_raw.columns:
    if c == "datetime":
        continue
    m1 = pat_tuple.match(c)
    if m1:
        t, f = m1.group(1), m1.group(2)
    else:
        m2 = pat_flat.match(c)
        if not m2:
            continue
        t, f = m2.group(1), m2.group(2)
    if f in ("close", "volume"):
        pairs.setdefault(t, {})[f] = c

tickers = sorted([t for t, d in pairs.items() if "close" in d and "volume" in d])
if MAX_TICKERS_VAL is not None:
    tickers = tickers[:MAX_TICKERS_VAL]
if not tickers:
    raise ValueError("validation.parquet 中未识别到 close/volume 成对列。")

val_long = pl.concat([
    val_raw.select([
        pl.col("datetime"),
        pl.lit(t).alias("ticker"),
        pl.col(pairs[t]["close"]).cast(pl.Float64).alias("close"),
        pl.col(pairs[t]["volume"]).cast(pl.Float64).alias("volume"),
    ])
    for t in tickers
], how="vertical_relaxed").sort(["ticker", "datetime"])

bars_per_hour = 12
w_1h, w_4h, w_8h, w_24h, w_3d = 12, 48, 96, 288, 864
eps = 1e-9

feat_val = val_long

# --- 先复现训练时来自 cell2 的关键链路（避免缺列）---
feat_val = feat_val.with_columns([
    (pl.col("close") / (pl.col("close").shift(1).over("ticker") + eps) - 1).alias("return"),
]).with_columns([
    pl.col("return").shift(1).over("ticker").alias("return_lag1"),
    pl.rolling_corr(pl.col("close"), pl.col("volume"), window_size=24).over("ticker").alias("close_volume_corr"),
]).with_columns([
    pl.col("close_volume_corr").rolling_mean(window_size=24).over("ticker").alias("corr_mean"),
    pl.col("close_volume_corr").rolling_std(window_size=24).over("ticker").alias("corr_std"),
]).with_columns([
    pl.col("corr_mean").rolling_mean(window_size=24).over("ticker").alias("corr_mean_ma"),
    pl.col("corr_std").rolling_mean(window_size=24).over("ticker").alias("corr_std_ma"),
])

# --- 复现训练时来自 cell3 的 5m 因子链路 ---
feat_val = feat_val.with_columns([
    (pl.col("close") / (pl.col("close").shift(1).over("ticker") + eps) - 1).alias("return_5m"),
    (pl.col("close") / (pl.col("close").shift(w_1h).over("ticker") + eps) - 1).alias("return_1h"),
    (pl.col("close") / (pl.col("close").shift(w_4h).over("ticker") + eps) - 1).alias("return_4h"),
    (pl.col("close") / (pl.col("close").shift(w_24h).over("ticker") + eps) - 1).alias("return_24h"),
    ((pl.col("volume") - pl.col("volume").shift(1).over("ticker")) / (pl.col("volume").shift(1).over("ticker") + eps)).fill_null(0.0).alias("volume_change_rate"),
]).with_columns([
    pl.when(pl.col("volume_change_rate") != 0)
    .then(pl.col("volume_change_rate").diff().over("ticker") / (pl.col("volume_change_rate") + eps))
    .otherwise(0.0)
    .alias("volume_change_rate_change_rate"),
    pl.col("volume").rank("dense").over("ticker").rolling_mean(window_size=w_24h).alias("volume_quantile_rank_24h"),
    pl.col("volume").rolling_std(window_size=w_8h).over("ticker").alias("volume_volatility_8h"),
    pl.col("volume").rolling_mean(window_size=w_4h).over("ticker").alias("volume_mean_4h"),
]).with_columns([
    pl.when(pl.col("return_5m") != 0)
    .then(pl.col("return_5m").diff().over("ticker") / (pl.col("return_5m") + eps))
    .otherwise(0.0)
    .alias("price_change_change_rate"),
    ((pl.col("return_5m") * pl.col("volume_change_rate")) >= 0).cast(pl.Int8).alias("price_volume_direction"),
    (pl.col("return_5m") * pl.col("volume_volatility_8h")).alias("momentum_volatility_factor"),
    (pl.col("close") > pl.col("close").rolling_max(window_size=w_24h).shift(1).over("ticker")).cast(pl.Int8).alias("price_breakout_confirmation"),
    (pl.col("volume") > pl.col("volume").rolling_max(window_size=w_24h).shift(1).over("ticker")).cast(pl.Int8).alias("volume_breakout_confirmation"),
]).with_columns([
    pl.col("return_5m").rolling_mean(window_size=w_24h).over("ticker").shift(1).alias("return_5m_24h_mean"),
    pl.col("return_1h").rolling_mean(window_size=w_24h).over("ticker").shift(1).alias("return_1h_24h_mean"),
    pl.col("return_5m").rolling_std(window_size=w_24h).over("ticker").shift(1).alias("return_5m_24h_vol"),
    pl.col("close").rolling_mean(window_size=w_8h).over("ticker").shift(1).alias("ma_close_past_8h"),
    pl.col("close").rolling_mean(window_size=w_4h).over("ticker").shift(1).alias("ma_close_past_4h"),
]).with_columns([
    pl.when(pl.col("ma_close_past_4h") != 0)
    .then(pl.col("ma_close_past_8h") / (pl.col("ma_close_past_4h") + eps) - 1)
    .otherwise(0.0)
    .alias("ma_ratio_fast_slow_past"),
])

feat_val = feat_val.with_columns([
    pl.when(pl.col("return_5m") > 0).then(pl.col("return_5m").rolling_mean(window_size=w_24h).over("ticker").shift(1)).otherwise(None).alias("upward_mean_past_24h"),
    pl.when(pl.col("return_5m") <= 0).then(pl.col("return_5m").rolling_mean(window_size=w_24h).over("ticker").shift(1)).otherwise(None).alias("downward_mean_past_24h"),
]).with_columns([
    pl.when(
        pl.col("upward_mean_past_24h").is_not_null() & pl.col("downward_mean_past_24h").is_not_null() & (pl.col("downward_mean_past_24h") != 0)
    ).then(pl.col("upward_mean_past_24h") / (pl.col("downward_mean_past_24h") + eps)).otherwise(0.0).alias("rs_past_24h"),
]).with_columns([
    pl.when(pl.col("rs_past_24h") != 0).then(100.0 - (100.0 / (pl.col("rs_past_24h") + eps))).otherwise(0.0).alias("rsi_past_24h"),
]).with_columns([
    (pl.col("rsi_past_24h") > 70).cast(pl.Int8).alias("rsi_overbought_past"),
    (pl.col("rsi_past_24h") < 30).cast(pl.Int8).alias("rsi_oversold_past"),
])

feat_val = feat_val.with_columns([
    pl.col("volume").diff().over("ticker").alias("volume_diff"),
    pl.col("volume").rolling_mean(window_size=w_24h).over("ticker").alias("volume_ma_24h"),
    pl.col("close").rolling_mean(window_size=w_24h).over("ticker").alias("close_ma_24h"),
]).with_columns([
    ((pl.col("volume_diff") / (pl.col("volume_ma_24h") + eps)).abs()).rolling_mean(window_size=w_24h).over("ticker").alias("factor_mo_05_01"),
    pl.col("volume_diff").rolling_std(window_size=w_24h).over("ticker").alias("volume_diff_std"),
    ((pl.col("close") - pl.col("close_ma_24h")) / (pl.col("close_ma_24h") + eps)).alias("close_change_rate"),
]).with_columns([
    pl.col("volume_diff_std").rolling_mean(window_size=w_24h).over("ticker").alias("factor_mo_05_02"),
    pl.col("close_change_rate").rolling_mean(window_size=w_24h).over("ticker").alias("factor_mo_04"),
    pl.rolling_corr(pl.col("close"), pl.col("volume"), window_size=w_24h).over("ticker").alias("close_volume_corr"),
]).with_columns([
    pl.col("close_volume_corr").rolling_mean(window_size=w_24h).over("ticker").alias("close_volume_corr_mean"),
    pl.col("close_volume_corr").rolling_std(window_size=w_24h).over("ticker").alias("close_volume_corr_std"),
]).with_columns([
    (
        (pl.col("close_volume_corr_mean") - pl.col("close_volume_corr_mean").rolling_mean(window_size=w_24h).over("ticker")) / (pl.col("close_volume_corr_std") + eps)
        + (pl.col("close_volume_corr_std") - pl.col("close_volume_corr_std").rolling_mean(window_size=w_24h).over("ticker")) / (pl.col("close_volume_corr_std").rolling_mean(window_size=w_24h).over("ticker") + eps)
    ).alias("factor_mo_03"),
    pl.col("return_5m").shift(1).over("ticker").alias("return_5m_lag1"),
]).with_columns([
    pl.rolling_corr(pl.col("return_5m"), pl.col("return_5m_lag1"), window_size=w_24h).over("ticker").alias("factor_mo_07"),
])

high_freq_cols = {
    "return_5m", "volume_change_rate", "volume_change_rate_change_rate",
    "price_change_change_rate", "price_volume_direction",
    "price_breakout_confirmation", "volume_breakout_confirmation"
}
exclude_cols = {
    "datetime", "ticker", "close", "volume", "return_5m_lag1", "close_ma", "volume_diff", "volume_ma", "volume_diff_std",
    "close_change_rate", "close_ma_24h", "volume_ma_24h", "close_volume_corr", "close_volume_corr_mean", "close_volume_corr_std",
    "upward_mean_past_24h", "downward_mean_past_24h", "rs_past_24h"
}

numeric_cols = [
    c for c, dt in feat_val.schema.items()
    if dt in (pl.Float32, pl.Float64, pl.Int8, pl.Int16, pl.Int32, pl.Int64) and c not in exclude_cols
]

ewm_base_cols = [c for c in numeric_cols if c not in high_freq_cols]
feat_val = feat_val.with_columns([
    pl.col(c).cast(pl.Float64).ewm_mean(span=w_3d, adjust=False, min_samples=1).over("ticker").alias(f"{c}_ewm3d")
    for c in ewm_base_cols
])

# 回测真实标签（5分钟前瞻收益）
feat_val = feat_val.with_columns([
    (pl.col("close").shift(-1).over("ticker") / (pl.col("close") + 1e-12) - 1).alias("target_fwd_5m")
])

print(f"feat_val shape: {feat_val.shape}")

In [ ]:
# [B] 预测（单独 cell）
from pathlib import Path
import joblib
import polars as pl

if "feat_val" not in globals():
    raise ValueError("请先运行 [A] 特征工程 cell。")

model_path = Path("artifacts/lgb_model_5m.pkl")
if not model_path.exists():
    raise FileNotFoundError(f"未找到模型文件: {model_path}")

bundle = joblib.load(model_path)
infer_model = bundle["model"]
trained_feature_cols = bundle["feature_cols"]
best_iter = bundle.get("best_iteration", None)

pred_input = feat_val

# 关键：把训练有但验证缺的列补齐，避免顺序/链路差异导致预测失败
missing_features = [c for c in trained_feature_cols if c not in pred_input.columns]
if missing_features:
    print(f"补齐缺失特征列数量: {len(missing_features)}")
    pred_input = pred_input.with_columns([
        pl.lit(None).cast(pl.Float64).alias(c) for c in missing_features
    ])

# 与训练一致的数据清理
pred_input = pred_input.with_columns([
    pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
    for c in trained_feature_cols
])

pred_input = pred_input.with_columns([
    pl.col(c).fill_null(pl.col(c).median().over("ticker")).fill_null(pl.col(c).median()).fill_null(0.0).cast(pl.Float32).alias(c)
    for c in trained_feature_cols
])

pred_arr = infer_model.predict(
    pred_input.select(trained_feature_cols).to_numpy(),
    num_iteration=best_iter if best_iter is not None else None
)

pred_df_val = pred_input.with_columns([
    pl.Series("pred_ret_5m", pred_arr)
]).filter(pl.col("target_fwd_5m").is_not_null())

print(f"pred_df_val shape: {pred_df_val.shape}")
print(pred_df_val.select(["datetime", "ticker", "pred_ret_5m", "target_fwd_5m"]).head(5))

In [ ]:
# [C] 回测 + 可视化 + 区间夏普（单独 cell）
import numpy as np
import polars as pl
import matplotlib.pyplot as plt

if "pred_df_val" not in globals():
    raise ValueError("请先运行 [B] 预测 cell。")

TOP_K = 1
bars_per_day = 78
annual_factor = np.sqrt(252 * bars_per_day)

# 每日尾盘平仓：若下一根bar跨日，则该时点不持仓（避免隔夜收益暴露）
pred_for_bt = (
    pred_df_val
    .sort(["ticker", "datetime"])
    .with_columns([
        pl.col("datetime").shift(-1).over("ticker").alias("next_dt"),
    ])
    .with_columns([
        (pl.col("next_dt").dt.date() == pl.col("datetime").dt.date())
        .fill_null(False)
        .alias("can_hold_to_next_bar")
    ])
)

# 先基于预测做截面TopK，再过滤掉跨日bar（尾盘平仓）
selected_df = (
    pred_for_bt
    .with_columns(pl.col("pred_ret_5m").rank("dense", descending=True).over("datetime").alias("pred_rank"))
    .filter((pl.col("pred_rank") <= TOP_K) & pl.col("can_hold_to_next_bar"))
)

# 按时间聚合组合收益
bt_raw = (
    selected_df
    .group_by("datetime")
    .agg([
        pl.col("target_fwd_5m").mean().alias("portfolio_ret_5m"),
        pl.len().alias("n_holdings"),
    ])
)

# 补齐所有时间点：无持仓时收益记0、持仓数记0（净值横盘）
full_time = pred_for_bt.select("datetime").unique().sort("datetime")

bt_df = (
    full_time
    .join(bt_raw, on="datetime", how="left")
    .with_columns([
        pl.col("portfolio_ret_5m").fill_null(0.0),
        pl.col("n_holdings").fill_null(0),
    ])
    .sort("datetime")
    .with_columns((pl.col("portfolio_ret_5m") + 1.0).cum_prod().alias("equity_curve"))
)

stat = bt_df.select([
    pl.col("portfolio_ret_5m").mean().alias("mu"),
    pl.col("portfolio_ret_5m").std().alias("sigma"),
    pl.col("equity_curve").last().alias("final_nav"),
    pl.col("n_holdings").mean().alias("avg_holdings"),
]).to_dicts()[0]

mu = float(stat["mu"]) if stat["mu"] is not None else 0.0
sigma = float(stat["sigma"]) if stat["sigma"] is not None else 0.0
sharpe = (mu / sigma * annual_factor) if sigma > 0 else np.nan

print("Validation 区间回测统计")
print(f"mean_5m_ret : {mu:.8f}")
print(f"std_5m_ret  : {sigma:.8f}")
print(f"ann_sharpe  : {sharpe:.4f}")
print(f"final_nav   : {float(stat['final_nav']):.6f}")
print(f"avg_holdings: {float(stat['avg_holdings']):.2f}")

plot_pd = bt_df.select(["datetime", "equity_curve"]).to_pandas()
plt.figure(figsize=(12, 4))
plt.plot(plot_pd["datetime"], plot_pd["equity_curve"], label=f"Top{TOP_K} Equal-Weight Long")
plt.title("Validation Equity Curve (Top15, 5m Rebalance)")
plt.xlabel("Datetime")
plt.ylabel("Net Value")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()